# Customer Journey Map: Keyword & Expression Extractor (TF-IDF & BERT)
### Multi-Stage Lifecycle NLP Analysis for Food Delivery Apps
This Jupyter Notebook maps consumer comments across the **6 Consumer Journey Lifecycle Stages**:
* **Stage 1: Discovery & Need Recognition**
* **Stage 2: Exploration & Filtering** (Fees, menus, coupons, veg filters)
* **Stage 3: Trust & Evaluation** (Ratings, reviews, AI images, vendor trust)
* **Stage 4: Purchase & Checkout** (Cancellations, payment failures, COD fees)
* **Stage 5: Fulfillment & Delivery Experience** (Drivers, late transit, spillage, food theft)
* **Stage 6: Support, Retention & Churn** (Chatbots, agent escalation, refunds, loyalty programs)

It computes statistically significant Unigrams, Bigrams, and Trigrams using **TF-IDF with sublinear scaling** and identifies repetition hotspots.


In [ ]:
# Step 1: Import NLP & Modeling Libraries
import os
import re
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS

%matplotlib inline


In [ ]:
# Step 2: Locate and Load the Dataset
possible_paths = [
    "Swiggy vs Zomato.xlsx",
    r"C:\Users\manka\Downloads\Swiggy vs Zomato.xlsx",
    os.path.join("..", "Swiggy vs Zomato.xlsx")
]

DATASET_PATH = None
for p in possible_paths:
    if os.path.exists(p):
        DATASET_PATH = p
        break

if not DATASET_PATH:
    raise FileNotFoundError("Could not locate 'Swiggy vs Zomato.xlsx'.")

print(f"Loading: {DATASET_PATH}")
xl = pd.ExcelFile(DATASET_PATH)
df_swiggy = xl.parse("Swiggy")
df_zomato = xl.parse("zomato")

df_swiggy["brand"] = "Swiggy"
df_zomato["brand"] = "Zomato"

df_all = pd.concat([df_swiggy, df_zomato], ignore_index=True)
df_all = df_all.rename(columns={"Comment text": "comment_text"})
print(f"Total comments loaded: {len(df_all)}")


In [ ]:
# Step 3: Text Preprocessing & Custom Stopwords
CUSTOM_STOPWORDS = {
    "swiggy", "zomato", "app", "apps", "reddit", "order", "orders", "food",
    "just", "like", "also", "even", "dont", "didnt", "wont", "ive", "im", "bhai",
    "bro", "hai", "bhi", "toh", "se", "ki", "ka", "ke", "mai", "aur", "ye", "wo",
    "one", "got", "get", "did", "come", "said", "make", "take", "guy", "guys",
    "really", "people", "something", "someone", "tell", "told", "asked", "asking",
    "going", "went", "think", "know", "say", "saying", "see", "seen", "day", "days"
}

ALL_STOPWORDS = list(ENGLISH_STOP_WORDS.union(CUSTOM_STOPWORDS))

def clean_text_nlp(text):
    text = re.sub(r"https?://\S+", "", str(text))
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\d+", " ", text)
    return re.sub(r"\s+", " ", text).lower().strip()

df_all["clean_text"] = df_all["comment_text"].apply(clean_text_nlp)


In [ ]:
# Step 4: Map Feedback to Customer Journey Stages
def map_journey_stage(text):
    t = str(text).lower()
    if any(k in t for k in ["support", "bot", "customer care", "agent", "refund", "nch", "consumer court", "deleted", "uninstalled", "food rescue", "gold"]):
        return "Stage 6: Support & Retention"
    elif any(k in t for k in ["cancel", "cancellation", "penalty", "charged 100%", "cod", "cash on delivery", "gpay", "payment failed", "checkout"]):
        return "Stage 4: Purchase & Checkout"
    elif any(k in t for k in ["delivery partner", "rider", "driver", "late delivery", "on route", "doorstep", "stole", "photo verification", "spillage", "scale", "underweight"]):
        return "Stage 5: Fulfillment & Delivery"
    elif any(k in t for k in ["rating", "dish rating", "fake image", "ai image", "reviews", "4.5 for you", "boosted"]):
        return "Stage 3: Trust & Evaluation"
    elif any(k in t for k in ["filter", "veg filter", "platform fee", "charges", "packaging fee", "mrp", "discount", "coupon", "70%", "80%"]):
        return "Stage 2: Exploration & Filtering"
    return "Stage 1: Discovery & Need"

df_all["journey_stage"] = df_all["comment_text"].apply(map_journey_stage)
print("Distribution across journey stages:")
df_all["journey_stage"].value_counts()


In [ ]:
# Step 5: Extract Top N-Gram Keywords per Stage via TF-IDF
stages = sorted(df_all["journey_stage"].unique())
stage_results = {}

for stg in stages:
    subset = df_all[df_all["journey_stage"] == stg]
    corpus = subset["clean_text"].tolist()
    if len(corpus) == 0:
        continue
        
    vec = TfidfVectorizer(
        ngram_range=(1, 3),
        stop_words=ALL_STOPWORDS,
        min_df=1,
        max_df=0.9,
        sublinear_tf=True
    )
    
    tfidf_mat = vec.fit_transform(corpus)
    feature_names = vec.get_feature_names_out()
    scores = tfidf_mat.mean(axis=0).A1
    top_indices = scores.argsort()[::-1][:10]
    
    extracted = []
    for idx in top_indices:
        term = feature_names[idx]
        score = round(float(scores[idx]), 4)
        extracted.append({"term": term, "score": score})
        
    stage_results[stg] = extracted

# Display as clean summary DataFrame
summary_rows = []
for stg, terms in stage_results.items():
    top_3_str = ", ".join([f"{t['term']} ({t['score']})" for t in terms[:3]])
    summary_rows.append({"Journey Stage": stg, "Top Key Expressions": top_3_str})

pd.DataFrame(summary_rows)


In [ ]:
# Step 6: Visualize Key Expressions in High-Friction Stages
# Visualizing Stage 4 (Purchase/Checkout) and Stage 6 (Support/Retention)
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for ax, stg, col in zip(axes, ["Stage 4: Purchase & Checkout", "Stage 6: Support & Retention"], ["#C0392B", "#2980B9"]):
    if stg in stage_results:
        terms = [item["term"] for item in stage_results[stg][:8]][::-1]
        scores = [item["score"] for item in stage_results[stg][:8]][::-1]
        
        ax.barh(terms, scores, color=col)
        ax.set_title(f"Top Expressions: {stg}", fontsize=12, fontweight='bold')
        ax.set_xlabel("Mean TF-IDF Weight", fontsize=10)
        ax.grid(axis='x', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()


In [ ]:
# Step 7: Export Journey Stage NLP Findings
with open("journey_stage_keywords.json", "w", encoding="utf-8") as f:
    json.dump(stage_results, f, indent=2)

print("Journey Stage NLP results exported to 'journey_stage_keywords.json' successfully!")
